# Notas — Aula 7: Métodos especiais (dunder)

Marco: o `Robo` da Aula 6 já protege `x`, `y`, `bateria` e `direcao` com `@property` —
mas continua imprimindo feio (`<__main__.Robo object at 0x...>`) e não sabe comparar,
medir ou ser percorrido. Hoje ele ganha `__repr__`/`__str__` (imprimir bem), e uma
classe nova, `Posicao`, ganha `__eq__` (comparar por valor). Depois, `Robo` ganha
`__len__` (medir a trajetória) e `__iter__` (percorrer com `for`), e uma classe
`Grade` ganha `__getitem__` (indexar com `[]`).

> ⚠️ **Antes de começar:** rode todas as células a partir do topo (**Run All**) — cada
> seção depende da anterior.

In [42]:
LADO_GRADE = 10

## `__repr__` — a representação para quem programa

Sem `__repr__`, `print(objeto)` mostra `<__main__.Classe object at 0x...>` — o
endereço de memória, sem informação útil. `__repr__` ensina o Python a mostrar algo
melhor: aparece em `print`, no eco do REPL, no debugger, e dentro de listas/tuplas/
dicts (que **sempre** usam `__repr__` dos itens, nunca `__str__`, mesmo que o objeto
tenha os dois). No robô, usamos isso para inspecionar `Robo` e `Posicao` sem acessar
atributo por atributo.

In [43]:
class Posicao:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Posicao({self.x}, {self.y})"


p1 = Posicao(3, 4)
print(p1)
print([p1, Posicao(0, 0)])

Posicao(3, 4)
[Posicao(3, 4), Posicao(0, 0)]


In [44]:
class Robo:
    def __init__(self, nome, x=0, y=0, direcao="LESTE"):
        self.nome = nome
        self.x = x
        self.y = y
        self.direcao = direcao

    def __repr__(self):
        return f"Robo({self.nome!r}, x={self.x}, y={self.y}, direcao={self.direcao!r})"


robo1 = Robo("Wall-E")
robo2 = Robo("R2D2")
print(robo1)
print([robo1, robo2])

Robo('Wall-E', x=0, y=0, direcao='LESTE')
[Robo('Wall-E', x=0, y=0, direcao='LESTE'), Robo('R2D2', x=0, y=0, direcao='LESTE')]


### Sua vez

Complete `__repr__` da classe `Comando` (`acao`, `valor`): deve devolver algo como
`Comando('AVANCAR', 3)` — use `!r` nos dois campos, para que o resultado leia como a
chamada que recria o objeto.

*Dica: mesmo formato do `Posicao.__repr__` acima, só que com dois campos.*

In [45]:
class Comando:
    def __init__(self, acao, valor):
        self.acao = acao
        self.valor = valor

    def __repr__(self):
        # TODO: devolva f"Comando({self.acao!r}, {self.valor!r})"
        return f"Comando({self.acao!r}, {self.valor})"
        pass


c1 = Comando("AVANCAR", 3)
try:
    print(c1)
except TypeError as erro:
    print(f"TypeError: {erro}")

Comando('AVANCAR', 3)


## `__str__` — a representação para humanos

`__str__` é opcional — quando existe, `print(objeto)` e `str(objeto)` passam a usá-lo
em vez de `__repr__`; sem ele, o *fallback* é sempre `__repr__` (foi o que já
funcionou na seção anterior). A diferença: `__repr__` é para quem programa (preciso,
sem ambiguidade); `__str__` é para quem lê o resultado sem saber Python. Dentro de
uma coleção, o Python **sempre** usa `__repr__`, nunca `__str__`.

In [46]:
class Posicao:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Posicao({self.x}, {self.y})"

    def __str__(self):
        return f"({self.x}, {self.y})"


p1 = Posicao(3, 4)
print(p1)         # usa __str__
print([p1])       # usa __repr__, mesmo com __str__ definido

(3, 4)
[Posicao(3, 4)]


In [47]:
class Robo:
    def __init__(self, nome, x=0, y=0, direcao="LESTE"):
        self.nome = nome
        self.x = x
        self.y = y
        self.direcao = direcao

    def __repr__(self):
        return f"Robo({self.nome!r}, x={self.x}, y={self.y}, direcao={self.direcao!r})"

    def __str__(self):
        return f"{self.nome} em ({self.x}, {self.y}), direção {self.direcao}"


robo1 = Robo("Wall-E")
print(robo1)
print([robo1])

Wall-E em (0, 0), direção LESTE
[Robo('Wall-E', x=0, y=0, direcao='LESTE')]


### Sua vez

Complete `__str__` da classe `Comando`: devolva uma versão legível, no formato
`"AVANCAR 3"` (ação e valor separados por espaço, sem aspas nem parênteses).

*Dica: um f-string com `self.acao` e `self.valor`, sem `!r` desta vez.*

In [48]:
class Comando:
    def __init__(self, acao, valor):
        self.acao = acao
        self.valor = valor

    def __repr__(self):
        return f"Comando({self.acao!r}, {self.valor!r})"

    def __str__(self):
        # TODO: devolva f"{self.acao} {self.valor}"
        pass


c1 = Comando("AVANCAR", 3)
try:
    print(c1)
except TypeError as erro:
    print(f"TypeError: {erro}")
print([c1])

TypeError: __str__ returned non-string (type NoneType)
[Comando('AVANCAR', 3)]


## `__eq__` — comparar por valor, não por identidade

Sem `__eq__`, `==` cai no comportamento padrão do Python: compara **identidade** (é o
mesmo objeto na memória?), igual `is` faria — dois objetos com os mesmos valores dão
`False`. `__eq__` ensina o Python a comparar pelo **conteúdo**. A versão segura sempre
começa checando `isinstance`: se o outro lado não for do mesmo tipo, devolve
`NotImplemented` (não é uma exceção) — o Python então tenta o caminho inverso antes de
desistir, em vez de estourar `AttributeError`.

In [49]:
class Posicao:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Posicao({self.x}, {self.y})"

    def __eq__(self, outra):
        if not isinstance(outra, Posicao):
            return NotImplemented
        return self.x == outra.x and self.y == outra.y


p1 = Posicao(3, 4)
p2 = Posicao(3, 4)
print(p1 == p2)         # True — mesmo conteúdo
print(p1 == (3, 4))     # False — não quebra, graças ao isinstance

True
False


In [50]:
def posicoes_iguais(robo_a, robo_b):
    return Posicao(robo_a.x, robo_a.y) == Posicao(robo_b.x, robo_b.y)


class RoboMini:
    def __init__(self, x, y):
        self.x = x
        self.y = y


ra = RoboMini(3, 4)
rb = RoboMini(3, 4)
rc = RoboMini(5, 5)
print(posicoes_iguais(ra, rb))
print(posicoes_iguais(ra, rc))

True
False


### Sua vez

Complete `__eq__` da classe `Comando`: se `outro` não for `Comando`, devolva
`NotImplemented`; senão, compare `acao` e `valor`.

*Dica: mesma estrutura do `Posicao.__eq__` acima, com dois campos em vez de um.*

In [51]:
class Comando:
    def __init__(self, acao, valor):
        self.acao = acao
        self.valor = valor

    def __repr__(self):
        return f"Comando({self.acao!r}, {self.valor!r})"

    def __eq__(self, outro):
        # TODO: se outro não for Comando, devolva NotImplemented;
        # senão, compare acao e valor
        pass


c1 = Comando("AVANCAR", 3)
c2 = Comando("AVANCAR", 3)
print(c1 == c2)
print(c1 == "AVANCAR")

None
None


## `__len__` — medir com `len()`

Sem `__len__`, `len(objeto)` dá `TypeError: object has no len()`. `__len__` devolve
um inteiro — sempre `>= 0` — que representa "quantos elementos" o objeto tem. No
robô, `len(robo)` mede quantas posições a `trajetoria` já visitou (diferente de
`len(robo.historico)`, que conta comandos executados — `GIRAR` entra no histórico mas
não move o robô, então os dois números divergem).

In [52]:
class Caixa:
    def __init__(self):
        self.itens = []

    def __len__(self):
        return len(self.itens)


caixa = Caixa()
caixa.itens.append("parafuso")
caixa.itens.append("porca")
print(len(caixa))

2


In [53]:
class Robo:
    DELTAS = {"LESTE": (1, 0), "NORTE": (0, 1), "OESTE": (-1, 0), "SUL": (0, -1)}

    def __init__(self, nome, x=0, y=0, direcao="LESTE"):
        self.nome = nome
        self.x = x
        self.y = y
        self.direcao = direcao
        self.trajetoria = [(x, y)]
        self.historico = []

    def __len__(self):
        return len(self.trajetoria)

    def avancar(self, passos):
        dx, dy = Robo.DELTAS[self.direcao]
        for _ in range(passos):
            self.x += dx
            self.y += dy
            self.trajetoria.append((self.x, self.y))
        self.historico.append(f"AVANCAR {passos}")

    def girar(self, lado):
        self.historico.append(f"GIRAR {lado}")


robo1 = Robo("Wall-E")
robo1.avancar(2)
robo1.girar("ESQ")
print(len(robo1))          # posições visitadas: inicial + 2 avanços
print(len(robo1.historico))  # comandos executados: AVANCAR, GIRAR

3
2


### Sua vez

Complete `__len__` da classe `Grade`: devolva o total de células
(`self.lado * self.lado`).

*Dica: não use `len()` de nada — é só uma multiplicação.*

In [54]:
class Grade:
    def __init__(self, lado, obstaculos=None):
        self.lado = lado
        self.obstaculos = obstaculos if obstaculos is not None else {}

    def __len__(self):
        # TODO: devolva self.lado * self.lado
        pass


grade = Grade(5, {(2, 2): True})
try:
    print(len(grade))
except TypeError as erro:
    print(f"TypeError: {erro}")

TypeError: 'NoneType' object cannot be interpreted as an integer


## `__iter__` — percorrer com `for`

Sem `__iter__`, `for x in objeto` dá `TypeError: object is not iterable`. `__iter__`
precisa devolver um **iterador** (não só algo iterável) — por isso `iter(self.lista)`
(que **cria** um iterador a partir da lista), e não `return self.lista` sozinho, que
devolveria a lista, não um iterador, e quebraria com `TypeError: iter() returned
non-iterator`. Como `iter()` cria um iterador novo a cada chamada, `robo1` nunca se
esgota — pode ser percorrido várias vezes.

In [55]:
class Robo:
    DELTAS = {"LESTE": (1, 0), "NORTE": (0, 1), "OESTE": (-1, 0), "SUL": (0, -1)}

    def __init__(self, nome, x=0, y=0, direcao="LESTE"):
        self.nome = nome
        self.x = x
        self.y = y
        self.direcao = direcao
        self.trajetoria = [(x, y)]

    def __iter__(self):
        return iter(self.trajetoria)

    def avancar(self):
        dx, dy = Robo.DELTAS[self.direcao]
        self.x += dx
        self.y += dy
        self.trajetoria.append((self.x, self.y))


robo1 = Robo("Wall-E")
robo1.avancar()
robo1.avancar()

for pos in robo1:
    print(pos)

(0, 0)
(1, 0)
(2, 0)


In [56]:
print(list(robo1))
for pos in robo1:      # de novo
    pass
print(list(robo1))     # de novo — ainda funciona, não se esgota

[(0, 0), (1, 0), (2, 0)]
[(0, 0), (1, 0), (2, 0)]


### Sua vez

Complete `__iter__` da classe `Frota`, que guarda uma lista de robôs em
`self.robos`: devolva um iterador sobre `self.robos`, para que `for robo in frota`
funcione.

*Dica: mesma receita do `Robo.__iter__` acima — `iter(...)` em volta da lista.*

In [57]:
class Frota:
    def __init__(self, robos):
        self.robos = robos

    def __iter__(self):
        # TODO: devolva iter(self.robos)
        pass


frota = Frota(["Wall-E", "R2D2", "Bender"])
try:
    for nome in frota:
        print(nome)
except TypeError as erro:
    print(f"TypeError: {erro}")

TypeError: iter() returned non-iterator of type 'NoneType'


## `__getitem__` — indexar com `[]`

`objeto[chave]` chama `objeto.__getitem__(chave)` por trás dos panos — a mesma
mecânica de açúcar sintático de `self` e `==`, agora para `[]`. Não é exclusivo de
lista/dict/tupla: qualquer classe que implemente `__getitem__` aceita `[]`. Nossa
`Grade` não valida nada sobre o índice — se a chave não estiver em
`self.obstaculos`, devolve `"."`, calado, mesmo para um índice sem sentido nenhum.

In [58]:
class Grade:
    def __init__(self, lado, obstaculos=None):
        self.lado = lado
        self.obstaculos = obstaculos if obstaculos is not None else {}

    def __getitem__(self, pos):
        if pos in self.obstaculos:
            return "X"
        return "."


grade = Grade(5, {(2, 2): True, (3, 1): True})
print(grade[(2, 2)])
print(grade[(0, 0)])

for y in range(5):
    linha = "".join(grade[(x, y)] for x in range(5))
    print(linha)

X
.
.....
...X.
..X..
.....
.....


In [59]:
print(grade[999])   # índice sem sentido — sem erro, só ".

.


### Sua vez

Complete `__contains__` da classe `Grade`, para que `(x, y) in grade` funcione sem
acessar `grade.obstaculos` na mão: devolva `True` se `pos` estiver em
`self.obstaculos`.

*Dica: uma linha — `pos in self.obstaculos`.*

In [60]:
class Grade:
    def __init__(self, lado, obstaculos=None):
        self.lado = lado
        self.obstaculos = obstaculos if obstaculos is not None else {}

    def __getitem__(self, pos):
        if pos in self.obstaculos:
            return "X"
        return "."

    def __contains__(self, pos):
        # TODO: devolva True se pos estiver em self.obstaculos
        pass


grade = Grade(5, {(2, 2): True})
print((2, 2) in grade)
print((0, 0) in grade)

False
False


## Para aprofundar

- Métodos especiais (visão geral) — Data model: https://docs.python.org/3/reference/datamodel.html#special-method-names
- `__repr__`/`__str__` — documentação oficial: https://docs.python.org/3/reference/datamodel.html#object.__repr__ · https://docs.python.org/3/reference/datamodel.html#object.__str__
- `__eq__` e `NotImplemented` — documentação oficial: https://docs.python.org/3/reference/datamodel.html#object.__eq__
- `__len__`/`__iter__`/iteradores — Tutorial oficial: https://docs.python.org/3/tutorial/classes.html#iterators
- `__getitem__`/`__contains__` — documentação oficial: https://docs.python.org/3/reference/datamodel.html#object.__getitem__